# Predictive Analytics for E-Learning Dropout Rates
**BCA Final Project — Galgotias University** · Yatharth Vardhan Gupta, Dipti Rani, Annu Chaudhary · Guide: Mr. Kuldeep Kumar

Notebook stages follow Section 4.4 of the report. Set `USE_DEMO = False` and point `DATA_PATH` at your **real** dataset before collecting results for the report.

## 1. Import libraries and configuration

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import pandas as pd
import matplotlib.pyplot as plt
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))          # lets the notebook import the project package

from dropout_prediction import pipeline as dp
from dropout_prediction.demo_data import generate

USE_DEMO   = True                      # False -> use your real dataset
DATA_PATH  = str(ROOT / "data" / "your_dataset.csv")   # real dataset (CSV/XLSX)
DATASET_NAME = "Your dataset name + citation"   # goes into the report
TARGET     = "dropout_status"
FEATURES   = dp.DEFAULT_FEATURES       # edit to match your dataset
OUT_DIR    = str(ROOT / "outputs")
dp.environment_info()                  # NFR1: record package versions

## 2. Load the dataset (FR1)
> With `USE_DEMO=True` the data is **synthetic** — it proves the code works, not anything about real learners.

In [ ]:
if USE_DEMO:
    df, name, synthetic = generate(), "SYNTHETIC demo data", True
else:
    df, name, synthetic = dp.load_dataset(DATA_PATH), DATASET_NAME, False
df.head()          # screenshot 1: dataset preview

## 3. Validate and summarise data quality (FR2, FR3)

In [ ]:
df = dp.validate_dataset(df, FEATURES, TARGET)
quality = dp.data_quality_summary(df, TARGET)
print(quality)
pd.Series(quality["missing_by_column"], name="missing").to_frame()   # screenshot 2

## 4. Clean data and engineer features (Sections 3.3, 3.5)

In [ ]:
df, n_dup = dp.remove_duplicates(df)
print("duplicates removed:", n_dup)
df = dp.add_engineered_features(df)
FEATURES_ALL = list(FEATURES) + ["interactions_per_login", "engagement_score"]

## 5. Exploratory data analysis (FR4)

In [ ]:
figs = dp.plot_eda(df, FEATURES_ALL, TARGET, f"{OUT_DIR}/figures")
from IPython.display import Image, display
for f in figs: display(Image(f))     # screenshot 3

## 6. Split, train and cross-validate (FR5, FR6)
Preprocessing lives inside the scikit-learn Pipeline, so it is fitted on training data only (no leakage).

In [ ]:
split = dp.split_data(df, FEATURES_ALL, TARGET)
fitted, cv_table = dp.train_models(split)
cv_table.round(4)                    # screenshot 4

## 7. Evaluate on the held-out test set (FR7)

In [ ]:
metrics, cms = dp.evaluate_models(fitted, split)
best = metrics.drop(index="Baseline (majority class)")["F1-score"].idxmax()
print("Best model by F1:", best)
metrics                              # screenshot 6 -> fills Table 4.3

In [ ]:
dp.plot_confusion_matrix(cms[best], f"Confusion matrix - {best}", f"{OUT_DIR}/figures/05_confusion_matrix_best.png")
display(Image(f"{OUT_DIR}/figures/05_confusion_matrix_best.png"))   # screenshot 5
dp.feature_importance(fitted[best], split.X_test, split.y_test).round(4)

## 8. Predict risk and export report (FR8, FR9)

In [ ]:
preds = dp.predict_risk(fitted[best], split.X_test)
preds.insert(0, "actual_status", split.y_test.map({0:"Completed", 1:"Dropout"}))
preds.sort_values("dropout_probability", ascending=False).head(10)

In [ ]:
imp = dp.feature_importance(fitted[best], split.X_test, split.y_test)
print(dp.export_report(OUT_DIR, name, quality, cv_table, metrics, cms, best, imp, synthetic))

> Predictions are estimates for offering support, never for penalising learners.